In [0]:
# =============================================================================
# bronze_common_fixes.py
# Overrides functions from bronze_common.py to fix known bugs.
# %run this AFTER %run ./bronze_common.py in every bronze notebook.
#
# Bug 1: checkpoint_loc += load_type (fixed in each notebook's path cell)
# Bug 2: MERGE guard now uses record_hash OR data_timestamp (below)
# Bug 3: data_timestamp removed from xxhash64 fingerprint (below)
# Bug 4: print_landing_zone_files() shows files before Auto Loader (below)
# Bug 5: Enhanced CDC output in process_bronze_batch (below)
# =============================================================================

# ── Bug 3 FIX: Remove data_timestamp from hash ──
# Old code included data_timestamp in the hash, meaning re-sends with
# identical timestamps were considered "unchanged" even if a field value
# changed. Now only actual data fields are hashed.
def get_hash_cols(df) -> list:
    core_attr = {
        "domain", "name", "email",
        "contact_source_system_object", "contact_source_key_value",
        "campaign_source_key_value", "event_timestamp",
        # "data_timestamp" deliberately EXCLUDED (Bug 3 fix)
    }
    return [c for c in df.columns if c.startswith("a_") or c in core_attr]

# ── Bug 4 FIX: Show landing zone files before Auto Loader ──
def print_landing_zone_files():
    """Lists parquet files in the landing zone so you can see what's there."""
    try:
        all_files = []
        def _walk(path):
            try:
                for item in dbutils.fs.ls(path):
                    if item.name.endswith(".parquet"):
                        all_files.append(item)
                    elif item.isDir():
                        _walk(item.path)
            except Exception:
                pass
        _walk(landing_zone_path)

        print(f"\n  -- Landing Zone Files ({object_name}/{load_type}) --")
        print(f"  Path: {landing_zone_path}")
        if all_files:
            print(f"  Found {len(all_files)} parquet file(s):")
            for f in all_files:
                size_kb = round(f.size / 1024, 1)
                print(f"    {f.name}  {size_kb} KB")
        else:
            print(f"  No parquet files found.")
        print()
        return len(all_files)
    except Exception as e:
        print(f"  Could not list landing zone: {e}")
        return 0

print("  Bug 3 + Bug 4 fixes loaded")

In [0]:
# =============================================================================
# Bug 2 FIX: MERGE guard now uses record_hash OR data_timestamp
# Bug 5 FIX: Enhanced CDC output showing changed record details
# =============================================================================
def process_bronze_batch(batch_df, batch_id):
    if batch_df.isEmpty():
        bronze_logger.log_batch(batch_id, skipped=True)
        return

    landing_count = batch_df.count()
    active_spark = batch_df.sparkSession

    # A. Normalize
    df_norm = normalize_batch(batch_df, source_system, object_name,
                              sf_object_name, tenant_id)

    # B. Intra-batch dedup
    dedup_keys = ["source_system", "source_system_object",
                  "source_key_name", "source_key_value"]
    if "extraction_timestamp" in df_norm.columns:
        df_deduped = (
            df_norm
            .sort(F.col("extraction_timestamp").desc(), F.col("data_timestamp").desc())
            .dropDuplicates(dedup_keys)
            .drop("extraction_timestamp")
        )
    else:
        df_deduped = df_norm.dropDuplicates(dedup_keys)

    # C. Record hash for CDC (uses Bug 3 fixed get_hash_cols)
    hash_cols = get_hash_cols(df_deduped)
    df_deduped = df_deduped.withColumn(
        "record_hash",
        F.xxhash64(*[F.col(c) for c in hash_cols]).cast("string")
        if hash_cols else F.lit(None).cast("string")
    )
    deduped_count = df_deduped.count()

    # D. Schema evolution
    try:
        evolve_schema_if_needed(df_deduped.schema, table_full_name)
    except Exception as e:
        print(f"  Schema evolution warning (non-fatal): {e}")

    # E. CDC: Cross-batch change detection
    try:
        target_hashes = (
            active_spark.table(table_full_name)
                        .select("id", F.col("record_hash").alias("t_hash"))
        )
        df_changed = (
            df_deduped
            .join(target_hashes, on="id", how="left")
            .filter(F.col("t_hash").isNull() | (F.col("record_hash") != F.col("t_hash")))
            .drop("t_hash")
        )
    except Exception:
        df_changed = df_deduped

    changed_count = df_changed.count()

    # Bug 5: Enhanced CDC output
    print(f"  CDC Analysis - Batch {batch_id}")
    print(f"  " + "-" * 50)
    print(f"  Records in this landing file  : {landing_count:,}")
    print(f"  After intra-batch dedup       : {deduped_count:,}")
    print(f"  CDC: Hash-changed / new       : {changed_count:,}")
    print(f"  CDC: Unchanged (skipped)      : {deduped_count - changed_count:,}")

    if changed_count == 0:
        bronze_logger.log_batch(batch_id, landing=landing_count,
                                deduped=deduped_count, merged=0)
        print(f"  Batch {batch_id}: CDC filtered all records - no changes to merge.")
        return

    # Bug 5: Show changed record details
    try:
        new_recs = df_changed.join(
            active_spark.table(table_full_name).select("id"), on="id", how="left_anti"
        ).count()
        upd_recs = changed_count - new_recs
        print(f"  NEW records      : {new_recs}")
        print(f"  UPDATED records  : {upd_recs}")
    except Exception:
        pass

    # F. COALESCE MERGE with Bug 2 FIX
    drop_cols = [c for c in df_changed.columns if c.startswith("_")]
    df_changed = df_changed.drop(*drop_cols)

    view_name = f"bronze_{object_name}_{batch_id}_{int(time.time())}"
    df_changed.createOrReplaceTempView(view_name)

    cols = df_changed.columns
    pk_cols = {"tenant", "id", "source_system", "source_system_object",
               "source_key_name", "source_key_value", "created_at", "status"}
    always_overwrite = {"updated_at", "data_timestamp", "record_hash"}

    update_parts = []
    for c in cols:
        if c in pk_cols:
            continue
        if c in always_overwrite:
            update_parts.append(f"target.`{c}` = source.`{c}`")
        else:
            update_parts.append(
                f"target.`{c}` = COALESCE(source.`{c}`, target.`{c}`)"
            )
    update_set = ", ".join(update_parts) + ", target.`status` = 'updated'"

    ins_cols = ", ".join(f"`{c}`" for c in cols)
    ins_vals = ", ".join(
        f"source.`{c}`" if c != "status" else "'new'" for c in cols
    )

    # Bug 2 FIX: MERGE guard uses record_hash OR data_timestamp
    active_spark.sql(f"""
        MERGE INTO {table_full_name} AS target
        USING {view_name} AS source
        ON  target.source_system        = source.source_system
        AND target.source_system_object = source.source_system_object
        AND target.source_key_name      = source.source_key_name
        AND target.source_key_value     = source.source_key_value
        WHEN MATCHED
          AND (
            source.data_timestamp > COALESCE(target.data_timestamp,
                CAST('1900-01-01' AS TIMESTAMP))
            OR source.record_hash != COALESCE(target.record_hash, '')
          )
        THEN UPDATE SET {update_set}
        WHEN NOT MATCHED THEN INSERT ({ins_cols}) VALUES ({ins_vals})
    """)

    bronze_logger.log_batch(batch_id, landing=landing_count,
                            deduped=deduped_count, merged=changed_count)
    print(f"  Batch {batch_id}: MERGE complete -> {table_full_name}")
    print(f"     {changed_count:,} record(s) written (CDC: only what changed)")

print("  Bug 2 + Bug 5 fixes loaded")

In [0]:
# =============================================================================
# Override start_ingestion: Landing zone listing + batch fallback
# =============================================================================
def start_ingestion():
    """Landing zone to bronze via foreachBatch.
    Bug 4: Lists files before Auto Loader so you can see what exists.
    Fallback: If Auto Loader returns 0 but files exist, batch-reads them.
    """
    bronze_logger.reset()
    print(f"Auto Loader: {landing_zone_path} -> {table_full_name}")

    # Bug 4: Show what files are in the landing zone BEFORE Auto Loader
    print_landing_zone_files()

    # Phase 1: Try Auto Loader (streaming)
    df_stream = (
        spark.readStream
             .format("cloudFiles")
             .option("cloudFiles.format",             "parquet")
             .option("cloudFiles.inferColumnTypes",    "true")
             .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
             .option("cloudFiles.schemaLocation",      schema_loc)
             .option("cloudFiles.maxFilesPerTrigger",  str(max_files_per_trigger))
             .option("cloudFiles.useNotifications",    "false")
             .load(landing_zone_path)
             .withColumn("extraction_timestamp",
                         F.col("_metadata.file_modification_time"))
    )
    query = (
        df_stream.writeStream
                 .foreachBatch(process_bronze_batch)
                 .option("checkpointLocation", checkpoint_loc)
                 .trigger(availableNow=True)
                 .start()
    )
    query.awaitTermination()

    # Phase 2: Batch fallback if Auto Loader returned 0
    if bronze_logger.total_landing == 0:
        try:
            df_batch = (spark.read
                        .option("recursiveFileLookup", "true")
                        .parquet(landing_zone_path))
            batch_count = df_batch.count()
            if batch_count > 0:
                print(f"\n  Auto Loader returned 0 -- batch fallback found {batch_count:,} records")
                print(f"  Processing {batch_count:,} records through CDC pipeline...")
                process_bronze_batch(df_batch, batch_id=0)
        except Exception:
            pass  # No parquet files or truly empty

    summary = bronze_logger.print_summary()
    print(f"  Stream complete: {object_name}")
    return summary

print("  start_ingestion() upgraded (landing zone listing + batch fallback)")
print("\n  bronze_common_fixes loaded - all 5 bugs patched")